# Biomedical Deep Research Agent Team

A four-agent pipeline built on the [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/) and Deep Research API, tailored for biomedical and clinical research tasks.

**Pipeline overview:**

1. **Triage Agent** — Decides whether the query needs clarification before research begins.
2. **Clarifying Agent** — Asks 2–3 structured follow-up questions (PICO framework) to sharpen the query.
3. **Instruction Builder Agent** — Rewrites the enriched query into a precise, structured research brief.
4. **Research Agent** (`o3-deep-research`) — Performs web-scale literature search, streams progress, and produces a structured clinical report.

> **When to use this pipeline:** Complex literature reviews, systematic synthesis, clinical evidence summaries, drug/target analysis. For simple fact lookups, a standard `openai.responses` call is faster and cheaper.

## 1. Setup

Install dependencies (run once):

In [1]:
%pip install --upgrade "openai>=1.88" "openai-agents>=0.0.19" python-dotenv pydantic

Note: you may need to restart the kernel to use updated packages.


### Imports & client configuration

The `OPENAI_AGENTS_DISABLE_TRACING` flag enables Zero Data Retention (ZDR) mode — no conversation data is stored on OpenAI servers. Remove it if you want automatic tracing in the OpenAI platform dashboard.

In [2]:
import os
import json
from typing import List, Dict, Optional

from dotenv import load_dotenv
from pydantic import BaseModel
from openai import AsyncOpenAI
from agents import Agent, Runner, WebSearchTool, RunConfig, set_default_openai_client

# Load OPENAI_API_KEY from .env if present
load_dotenv()

client = AsyncOpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),
    timeout=600.0,  # Deep Research can take several minutes
)
set_default_openai_client(client)

# Disable tracing for Zero Data Retention environments
# Remove or set to "0" if you want traces visible in the OpenAI dashboard
os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"

print("Client configured. API key present:", bool(os.environ.get("OPENAI_API_KEY")))

Client configured. API key present: True


---

## 2. Prompts

Each supporting agent has a carefully designed system prompt to maximise the quality of the final research output.

In [3]:
# ─────────────────────────────────────────────────────────────
#  Triage Agent
# ─────────────────────────────────────────────────────────────
TRIAGE_AGENT_PROMPT = """
You are a biomedical research triage specialist. Evaluate the user's research query and decide whether
additional clarification is needed before deep research can be conducted.

Route to the Clarifying Questions Agent if ANY of the following are true:
- The patient population, disease, or condition is not clearly specified
- The intervention, drug, or therapy of interest is ambiguous
- The desired outcome measure or endpoint is unclear
- The query is too broad to yield a focused research report
- The time horizon or study design preference is absent and would materially affect results

Route directly to the Research Instruction Agent if:
- The query is specific and contains sufficient PICO elements (Population, Intervention, Comparator, Outcome)
- The research scope is well-defined and actionable

Return EXACTLY ONE function call:
• transfer_to_clarifying_questions_agent — if clarification is needed
• transfer_to_research_instruction_agent — if the query is ready for research
"""

# ─────────────────────────────────────────────────────────────
#  Clarifying Agent
# ─────────────────────────────────────────────────────────────
CLARIFYING_AGENT_PROMPT = """
You are a biomedical research clarification specialist. Your goal is to gather precisely the information
needed to conduct focused, high-quality clinical or scientific research.

Ask 2–3 targeted clarifying questions using the PICO framework as a guide:
  P — Population / Patient / Problem
  I — Intervention / Exposure / Test
  C — Comparator / Control (if relevant)
  O — Outcome / Endpoint of interest

GUIDELINES:
1. **Be specific and clinically grounded** — Ask about patient demographics, disease stage, line of therapy,
   specific biomarkers, or regulatory context where relevant.
2. **Do not ask for information already provided** — Only ask about missing dimensions.
3. **Be concise and professional** — Use numbered questions. Avoid condescending or overly verbose phrasing.
4. **Tailor to context** — For drug research, ask about mechanism class or indication. For genomics,
   ask about variant type or cancer type. For clinical trials, ask about phase or endpoint type.
"""

# ─────────────────────────────────────────────────────────────
#  Research Instruction Builder Agent
# ─────────────────────────────────────────────────────────────
RESEARCH_INSTRUCTION_AGENT_PROMPT = """
You are a biomedical research instruction architect. Take the user's query (and any clarification answers)
and rewrite it into a detailed, structured research brief for a Deep Research model.

OUTPUT ONLY THE RESEARCH INSTRUCTIONS — no preamble, no explanation. Then transfer to the research agent.

GUIDELINES:
1. **Maximize specificity** — Include all known PICO elements, patient population details, disease stage,
   biomarkers, and any constraints from the user.

2. **Fill unstated dimensions as open-ended** — If the user has not specified a comparator or time horizon,
   state explicitly: "Comparator: any / no specific constraint."

3. **Avoid unwarranted assumptions** — Do not invent details not provided by the user.

4. **Request structured output format** — Always ask the Deep Research model to format the final report with:
   - Executive Summary
   - Background & Disease Context
   - Evidence Review (organised by study type: RCTs, observational, meta-analyses)
   - Key Data Tables (efficacy, safety, biomarker associations — where relevant)
   - Clinical Implications
   - Limitations & Evidence Gaps
   - References (with inline URL citations)

5. **Prioritise high-quality sources** — Specify: PubMed/MEDLINE, ClinicalTrials.gov, FDA/EMA label data,
   peer-reviewed journals (NEJM, Lancet, JAMA, Nature Medicine, Cancer Cell, etc.).
   Prefer original publications over review aggregators.

6. **Evidence hierarchy** — Instruct the model to label evidence by study design
   (RCT > prospective cohort > retrospective > case series > expert opinion).

7. **Language** — Respond in English unless the user's query was in another language.

IMPORTANT: The complete payload must be valid for transfer to the research agent.
"""

---

## 3. Structured Output Schema

The Clarifying Agent returns a structured `Clarifications` object so that the pipeline can programmatically inject the user's answers back into the conversation.

In [4]:
class Clarifications(BaseModel):
    """Structured output from the Clarifying Agent."""
    questions: List[str]

---

## 4. Agent Definitions

The four agents are wired together via `handoffs` — each agent can transfer control to the next in the pipeline.

> **Model choice:**
> - `o4-mini-deep-research-2025-06-26` — Fast, cost-efficient. Suitable for most queries.
> - `o3-deep-research-2025-06-26` — Highest quality. Recommended for systematic reviews or complex synthesis.
>
> Change `RESEARCH_MODEL` below to switch between them.

In [5]:
# ── Model selection ────────────────────────────────────────────
RESEARCH_MODEL = "o4-mini-deep-research-2025-06-26"  # swap to o3-deep-research-2025-06-26 for max quality

# ── 1. Research Agent (leaf — no handoffs) ────────────────────
research_agent = Agent(
    name="Biomedical Research Agent",
    model=RESEARCH_MODEL,
    instructions=(
        "You are a senior biomedical research scientist. "
        "Perform rigorous, web-scale empirical research based on the provided research brief. "
        "Prioritise peer-reviewed literature, clinical trial registries, and regulatory sources. "
        "Cite all sources inline with URL citations."
    ),
    tools=[WebSearchTool()],
)

# ── 2. Research Instruction Builder Agent ─────────────────────
instruction_agent = Agent(
    name="Research Instruction Agent",
    model="gpt-4o-mini",
    instructions=RESEARCH_INSTRUCTION_AGENT_PROMPT,
    handoffs=[research_agent],
)

# ── 3. Clarifying Questions Agent ─────────────────────────────
clarifying_agent = Agent(
    name="Clarifying Questions Agent",
    model="gpt-4o-mini",
    instructions=CLARIFYING_AGENT_PROMPT,
    output_type=Clarifications,
    handoffs=[instruction_agent],
)

# ── 4. Triage Agent (entry point) ─────────────────────────────
triage_agent = Agent(
    name="Triage Agent",
    model="gpt-4o-mini",
    instructions=TRIAGE_AGENT_PROMPT,
    handoffs=[clarifying_agent, instruction_agent],
)

print("Agents defined:")
for agent in [triage_agent, clarifying_agent, instruction_agent, research_agent]:
    print(f"  • {agent.name} ({agent.model})")

Agents defined:
  • Triage Agent (gpt-4o-mini)
  • Clarifying Questions Agent (gpt-4o-mini)
  • Research Instruction Agent (gpt-4o-mini)
  • Biomedical Research Agent (o4-mini-deep-research-2025-06-26)


---

## 5. Research Runner

The `run_research` function orchestrates the full pipeline:

1. Starts the stream from the Triage Agent.
2. Detects when the Clarifying Agent emits questions → injects `mock_answers` (or `"No preference."`) back.
3. Streams and prints intermediate search queries from the Research Agent for transparency.
4. Returns the full stream object (containing `final_output` and `new_items` for post-processing).

**Using `mock_answers`:** Pass a dict mapping each clarification question string to your preferred answer. Any unanswered questions default to `"No preference."` This lets you either answer interactively (by extending the function) or pre-supply answers programmatically.

In [6]:
async def run_research(
    query: str,
    mock_answers: Optional[Dict[str, str]] = None,
    verbose: bool = False,
):
    """
    Run the full 4-agent biomedical research pipeline.

    Args:
        query:        The biomedical research question.
        mock_answers: Optional dict mapping clarification questions → answers.
                      Unanswered questions default to 'No preference.'.
        verbose:      If True, print all streaming events.

    Returns:
        The completed stream object (stream.final_output contains the report).
    """
    print(f"\n{'='*60}")
    print(f"QUERY: {query}")
    print(f"{'='*60}\n")

    stream = Runner.run_streamed(
        triage_agent,
        query,
        run_config=RunConfig(tracing_disabled=True),
    )

    async for ev in stream.stream_events():
        # ── Agent handoff notifications ──────────────────────
        if ev.type == "agent_updated_stream_event":
            print(f"\n--- Agent: {ev.new_agent.name} ---")

        # ── Clarification questions → inject answers ─────────
        elif isinstance(getattr(ev, "item", None), Clarifications):
            print("\n[Clarifying Agent] Questions:")
            reply_parts = []
            for q in ev.item.questions:
                ans = (mock_answers or {}).get(q, "No preference.")
                print(f"  Q: {q}")
                print(f"  A: {ans}")
                reply_parts.append(f"**{q}**\n{ans}")
            stream.send_user_message("\n\n".join(reply_parts))
            continue

        # ── Live web search queries from Research Agent ──────
        elif (
            ev.type == "raw_response_event"
            and hasattr(ev.data, "item")
            and hasattr(ev.data.item, "action")
        ):
            action = ev.data.item.action or {}
            if action.get("type") == "search":
                print(f"  [Web Search] {action.get('query')!r}")

        # ── Verbose mode: print all events ───────────────────
        elif verbose:
            print(ev)

    print("\n[Research complete]")
    return stream

---

## 6. Example Run

Modify the `query` and `mock_answers` below. If `mock_answers` is empty (`{}`), all clarification questions will be answered with `"No preference."` — the Research Agent will then treat those dimensions as unconstrained.

**Example biomedical queries to try:**
- `"What is the efficacy of CDK4/6 inhibitors in HR+/HER2- metastatic breast cancer?"`
- `"Summarise the current evidence for immunotherapy in microsatellite-stable colorectal cancer"`
- `"What are the resistance mechanisms to EGFR inhibitors in non-small cell lung cancer?"`
- `"Compare KRAS G12C inhibitors in clinical trials for pancreatic ductal adenocarcinoma"`

In [7]:
query = "What is the efficacy and safety of CDK4/6 inhibitors in HR+/HER2- metastatic breast cancer?"

# Optional: pre-supply answers to clarification questions.
# Keys must exactly match the question strings the Clarifying Agent produces.
# Leave empty ({}) to let the pipeline use 'No preference.' defaults.
mock_answers = {
    # Example — only active if the Clarifying Agent asks this exact question:
    # "What line of therapy are you interested in (first-line, second-line, or later)?": "First-line",
}

result = await run_research(query, mock_answers=mock_answers)

Tool name 'transfer_to_Clarifying Questions Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_clarifying_questions_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Research Instruction Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_research_instruction_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Clarifying Questions Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_clarifying_questions_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Research Instruction Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_research_instruction_agent'. Please use only letters, digits, and underscores to avoid pote


QUERY: What is the efficacy and safety of CDK4/6 inhibitors in HR+/HER2- metastatic breast cancer?


--- Agent: Triage Agent ---


Tool name 'transfer_to_Biomedical Research Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_biomedical_research_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Biomedical Research Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_biomedical_research_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.



--- Agent: Research Instruction Agent ---

--- Agent: Biomedical Research Agent ---


Error streaming response: Your organization must be verified to use the model `o4-mini-deep-research-2025-06-26`. Please go to: https://platform.openai.com/settings/organization/general and click on Verify Organization. If you just verified, it can take up to 15 minutes for access to propagate.


APIError: Your organization must be verified to use the model `o4-mini-deep-research-2025-06-26`. Please go to: https://platform.openai.com/settings/organization/general and click on Verify Organization. If you just verified, it can take up to 15 minutes for access to propagate.

---

## 7. Research Report

Print the final structured report:

In [ ]:
from IPython.display import display, Markdown

if result.final_output:
    display(Markdown(result.final_output))
else:
    print("No final output — check the agent interaction flow below for details.")

---

## 8. Agent Interaction Flow

Prints a human-readable sequence of every agent step: handoffs, tool calls, reasoning steps, and message outputs.

In [ ]:
def print_agent_interaction_flow(stream):
    """Print a numbered log of every agent action in the completed stream."""
    print("=== Agent Interaction Flow ===")
    count = 1

    for item in stream.new_items:
        agent_name = (
            getattr(item.agent, "name", "Unknown Agent")
            if hasattr(item, "agent")
            else "Unknown Agent"
        )

        if item.type == "handoff_call_item":
            func_name = getattr(item.raw_item, "name", "unknown")
            print(f"{count:>3}. [{agent_name}] → Handoff: {func_name}")

        elif item.type == "handoff_output_item":
            print(f"{count:>3}. [{agent_name}] → Handoff output received")

        elif item.type == "reasoning_item":
            print(f"{count:>3}. [{agent_name}] → Reasoning step")

        elif item.type == "tool_call_item":
            tool_name = getattr(item.raw_item, "name", None)
            if not isinstance(tool_name, str) or not tool_name.strip():
                continue
            tool_name = tool_name.strip()
            args = getattr(item.raw_item, "arguments", None)
            args_str = ""
            if args:
                try:
                    parsed = json.loads(args)
                    if parsed:
                        args_str = json.dumps(parsed)
                except Exception:
                    if args.strip() and args.strip() != "{}":
                        args_str = args.strip()
            suffix = f" | args: {args_str}" if args_str else ""
            print(f"{count:>3}. [{agent_name}] → Tool: {tool_name}{suffix}")

        elif item.type == "message_output_item":
            print(f"{count:>3}. [{agent_name}] → Message output")

        elif item.type == "mcp_list_tools_item":
            print(f"{count:>3}. [{agent_name}] → MCP tool list")

        else:
            print(f"{count:>3}. [{agent_name}] → {item.type}")

        count += 1

print_agent_interaction_flow(result)

---

## 9. Citation Extraction

Extracts all URL citations from the final report output, including the surrounding text for context.

In [ ]:
def print_citations(stream, context_chars: int = 80):
    """
    Extract and print all URL citations from the final research report.

    Args:
        stream:        Completed stream object from run_research().
        context_chars: Characters of preceding text to show for each citation.
    """
    citations_found = False

    for item in reversed(stream.new_items):
        if item.type != "message_output_item":
            continue

        for content in getattr(item.raw_item, "content", []):
            if not hasattr(content, "annotations") or not hasattr(content, "text"):
                continue

            text = content.text
            annotations = [
                ann
                for ann in content.annotations
                if getattr(ann, "type", None) == "url_citation"
            ]

            if not annotations:
                continue

            print(f"Found {len(annotations)} citation(s):\n")
            for i, ann in enumerate(annotations, 1):
                title = getattr(ann, "title", "<no title>")
                url = getattr(ann, "url", "<no url>")
                start = getattr(ann, "start_index", None)
                end = getattr(ann, "end_index", None)

                print(f"[{i}] {title}")
                print(f"     URL: {url}")

                if start is not None and end is not None and isinstance(text, str):
                    pre_start = max(0, start - context_chars)
                    preceding = text[pre_start:start].replace("\n", " ").strip()
                    excerpt = text[start:end].replace("\n", " ").strip()
                    if preceding:
                        print(f"     Context: '...{preceding}'")
                    if excerpt:
                        print(f"     Excerpt: '{excerpt}'")
                print()

            citations_found = True
        break  # Only process the last message_output_item

    if not citations_found:
        print("No URL citations found in the final output.")


print_citations(result)

---

## 10. Save Report to File

Optionally save the final Markdown report to disk:

In [ ]:
import re
from datetime import datetime

def save_report(stream, query: str, output_dir: str = "."):
    """
    Save the final research report as a Markdown file.

    Returns the file path, or None if no output was produced.
    """
    if not stream.final_output:
        print("No report to save.")
        return None

    # Sanitise the query into a safe filename
    slug = re.sub(r"[^\w\s-]", "", query.lower())[:60]
    slug = re.sub(r"[\s]+", "_", slug).strip("_")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{output_dir}/report_{timestamp}_{slug}.md"

    with open(filename, "w", encoding="utf-8") as f:
        f.write(f"# Research Report\n\n")
        f.write(f"**Query:** {query}\n\n")
        f.write(f"**Generated:** {datetime.now().isoformat()}\n\n")
        f.write(f"**Model:** {RESEARCH_MODEL}\n\n")
        f.write("---\n\n")
        f.write(stream.final_output)

    print(f"Report saved to: {filename}")
    return filename


# Uncomment to save:
# save_report(result, query)

---

## 11. Batch Research

Run multiple queries sequentially and save each report:

In [ ]:
async def run_batch_research(queries: List[str], output_dir: str = "."):
    """
    Run a list of biomedical research queries sequentially.
    Saves each report to a Markdown file.

    Returns a list of (query, stream) tuples.
    """
    results = []
    for i, q in enumerate(queries, 1):
        print(f"\n{'#'*60}")
        print(f"Running query {i}/{len(queries)}")
        stream = await run_research(q)
        save_report(stream, q, output_dir)
        results.append((q, stream))
    return results


# Example batch run (uncomment to execute):
# batch_queries = [
#     "Summarise PD-1/PD-L1 inhibitor efficacy in microsatellite-stable colorectal cancer",
#     "What are resistance mechanisms to osimertinib in EGFR-mutant NSCLC?",
#     "Compare KRAS G12C inhibitors (sotorasib vs adagrasib) in NSCLC clinical trials",
# ]
# batch_results = await run_batch_research(batch_queries)

---

## Next Steps

- **Add internal document search** — Connect an [MCP server](https://cookbook.openai.com/examples/deep_research_api/how_to_build_a_deep_research_mcp_server/readme) to query your own literature library alongside web search. Add `HostedMCPTool(tool_config={...})` to `research_agent.tools`.
- **Interactive clarification** — Replace `mock_answers` with a live `input()` loop or a UI widget to answer clarifying questions in real time.
- **Upgrade the model** — Switch `RESEARCH_MODEL` to `o3-deep-research-2025-06-26` for highest-quality systematic review synthesis.
- **Structured report parsing** — Add a post-processing agent that extracts structured data (tables, p-values, HR/OR) from the final Markdown report into a Pydantic model.